# 04 — Qualidade e exploração inicial da camada gold

Este notebook valida as tabelas da gold antes de qualquer análise e faz as primeiras explorações.

**Tabelas lidas:**
- `fato_importacao_mensal` (todas as partições `ano_mes=`)
- `dim_ncm` (uma linha por versão do NCM)
- `dim_pais`
- `dim_tempo`

**Premissas que os gráficos respeitam:**
- Os dados cobrem apenas combinações com ao menos 4 importadores por mês. Os valores **não são o total das importações do Brasil**.
- Métricas não aditivas (média, mediana, quartis, mínimo, máximo) **nunca são somadas**. Preço médio é sempre `SUM(vmle) / SUM(peso)`.
- Valores em US$ (dólar), peso em kg.

**Estrutura:**
1. Carga
2. Checagens de qualidade (relatório ✅ / ⚠️ / ❌)
3. Reconciliação com a silver
4. Exploração inicial (gráficos)

## 1 — Acesso e carga

In [ ]:
import io
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import yaml

with open('../../config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

bucket = cfg['aws']['bucket']
prefixo_gold = cfg['s3']['gold']
prefixo_silver = cfg['s3']['silver']
session = boto3.Session(profile_name=cfg['aws']['profile'], region_name=cfg['aws']['regiao'])
s3 = session.client('s3')


def listar(prefixo):
    chaves = []
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket, Prefix=prefixo):
        chaves += [o['Key'] for o in page.get('Contents', []) if o['Key'].endswith('.parquet')]
    return chaves


def ler(chave, colunas=None):
    corpo = s3.get_object(Bucket=bucket, Key=chave)['Body'].read()
    return pq.read_table(io.BytesIO(corpo), columns=colunas).to_pandas()


def ler_dim(nome):
    return ler(listar(f"{prefixo_gold}{nome}/")[0])


def ler_fato():
    return pd.concat([ler(c) for c in listar(f"{prefixo_gold}fato_importacao_mensal/")],
                     ignore_index=True)


def silver_vmle_por_mes():
    """Soma de vmle por mês direto da silver, para reconciliação."""
    registros = []
    for chave in listar(prefixo_silver):
        ano_mes = chave.split('ano_mes=')[1][:7]
        soma = ler(chave, ['vmle_soma_dolar_truncado'])['vmle_soma_dolar_truncado'].sum()
        registros.append({'ano_mes': ano_mes, 'vmle_silver': soma})
    return pd.DataFrame(registros).groupby('ano_mes', as_index=False)['vmle_silver'].sum()


fato = ler_fato()
dim_ncm = ler_dim('dim_ncm')
dim_pais = ler_dim('dim_pais')
dim_tempo = ler_dim('dim_tempo')

CHAVES = ['ano_mes', 'ncm', 'nves', 'pais_origem']
ADITIVAS = ['vmle_soma_dolar_truncado', 'peso_liq_soma_truncado', 'qtd_ume_soma_truncado']
METRICAS = [c for c in fato.columns if c.startswith(('fob_', 'frete_', 'seguro_', 'vmle_', 'peso_', 'qtd_'))]

print(f"fato        : {len(fato):,} linhas | {fato['ano_mes'].nunique()} meses | {fato['pais_origem'].nunique()} países")
print(f"dim_ncm     : {len(dim_ncm):,} versões | {dim_ncm['ncm'].nunique():,} NCMs")
print(f"dim_pais    : {len(dim_pais)} países")
print(f"dim_tempo   : {len(dim_tempo)} meses")
print(f"métricas    : {len(METRICAS)} ({len(ADITIVAS)} aditivas)")

## 2 — Checagens de qualidade

Cada checagem gera um resultado: ✅ passou, ⚠️ aviso (não impede a análise, mas precisa ser conhecido), ❌ erro (a tabela não deve ser usada até corrigir).

In [ ]:
resultados = []


def registrar(nivel, grupo, descricao, detalhe=''):
    resultados.append({'nivel': nivel, 'grupo': grupo, 'checagem': descricao, 'detalhe': detalhe})


# ── Estrutura ──────────────────────────────────────────────────────────────────
obrigatorias = CHAVES + ['ncm_versao_id', 'ume'] + ADITIVAS
faltam = [c for c in obrigatorias if c not in fato.columns]
if faltam:
    registrar('❌', 'Estrutura', 'Colunas obrigatórias presentes', str(faltam))
else:
    registrar('✅', 'Estrutura', 'Colunas obrigatórias presentes')

# ── Grão ───────────────────────────────────────────────────────────────────────
dup = fato.duplicated(CHAVES).sum()
if dup:
    registrar('❌', 'Grão', 'Chave única (ano_mes, ncm, nves, pais_origem)', f'{dup} duplicata(s)')
else:
    registrar('✅', 'Grão', 'Chave única (ano_mes, ncm, nves, pais_origem)')

# ── Integridade referencial ───────────────────────────────────────────────────
sem_versao = (~fato['ncm_versao_id'].isin(dim_ncm['ncm_versao_id'])).sum()
if sem_versao:
    registrar('❌', 'Integridade', 'ncm_versao_id existe em dim_ncm', f'{sem_versao} linha(s)')
else:
    registrar('✅', 'Integridade', 'ncm_versao_id existe em dim_ncm')

ncm_da_versao = dim_ncm.set_index('ncm_versao_id')['ncm']
ncm_incoerente = (fato['ncm_versao_id'].map(ncm_da_versao) != fato['ncm']).sum()
if ncm_incoerente:
    registrar('❌', 'Integridade', 'NCM da fato coincide com o NCM da versão', f'{ncm_incoerente} linha(s)')
else:
    registrar('✅', 'Integridade', 'NCM da fato coincide com o NCM da versão')

pais_sem = (~fato['pais_origem'].isin(dim_pais['pais_origem_rfb'])).sum()
if pais_sem:
    registrar('❌', 'Integridade', 'pais_origem existe em dim_pais', f'{pais_sem} linha(s)')
else:
    registrar('✅', 'Integridade', 'pais_origem existe em dim_pais')

if dim_ncm['ncm_versao_id'].duplicated().any():
    registrar('❌', 'Integridade', 'ncm_versao_id único em dim_ncm')
else:
    registrar('✅', 'Integridade', 'ncm_versao_id único em dim_ncm')

# ── Vigência: a versão usada cobre o último dia do mês ───────────────────────
ini = pd.to_datetime(dim_ncm.set_index('ncm_versao_id')['vigencia_inicio'])
fim = pd.to_datetime(dim_ncm.set_index('ncm_versao_id')['vigencia_fim'])
ref = pd.to_datetime(fato['ano_mes'] + '-01') + pd.offsets.MonthEnd(0)
v_ini = fato['ncm_versao_id'].map(ini)
v_fim = fato['ncm_versao_id'].map(fim)
fora = ((v_ini > ref) | (v_fim < ref)).sum()
if fora:
    registrar('❌', 'Vigência', 'Versão do NCM vigente no último dia do mês', f'{fora} linha(s)')
else:
    registrar('✅', 'Vigência', 'Versão do NCM vigente no último dia do mês')

# ── Cobertura temporal ────────────────────────────────────────────────────────
meses_fato = set(fato['ano_mes'])
meses_tempo = set(dim_tempo['ano_mes'])
fora_tempo = meses_fato - meses_tempo
if fora_tempo:
    registrar('❌', 'Cobertura', 'Meses da fato existem em dim_tempo', str(sorted(fora_tempo)))
else:
    registrar('✅', 'Cobertura', 'Meses da fato existem em dim_tempo')

sem_fato = sorted(meses_tempo - meses_fato)
if sem_fato:
    registrar('⚠️', 'Cobertura', 'Meses de dim_tempo sem dados na fato', f'{len(sem_fato)}: {sem_fato[:6]}...')
else:
    registrar('✅', 'Cobertura', 'Todos os meses de dim_tempo têm dados')

# ── Valores ───────────────────────────────────────────────────────────────────
for col in ['vmle_soma_dolar_truncado', 'peso_liq_soma_truncado', 'qtd_ume_soma_truncado']:
    neg = (fato[col] < 0).sum()
    if neg:
        registrar('❌', 'Valores', f'{col} não negativo', f'{neg} linha(s)')
    else:
        registrar('✅', 'Valores', f'{col} não negativo')

pares = [('fob_kg_minimo_dolar_truncado', 'fob_kg_maximo_dolar_truncado'),
         ('fob_ume_minimo_dolar_truncado', 'fob_ume_maximo_dolar_truncado'),
         ('frete_minimo_dolar_truncado', 'frete_maximo_dolar_truncado')]
for a, b in pares:
    if a in fato.columns and b in fato.columns:
        bad = ((fato[a] > fato[b]) & fato[a].notna() & fato[b].notna()).sum()
        if bad:
            registrar('❌', 'Valores', f'{a} <= {b}', f'{bad} linha(s)')
        else:
            registrar('✅', 'Valores', f'{a} <= {b}')

nulos_criticos = {c: int(fato[c].isna().sum()) for c in ADITIVAS if fato[c].isna().any()}
if nulos_criticos:
    registrar('❌', 'Valores', 'Métricas aditivas sem nulos', str(nulos_criticos))
else:
    registrar('✅', 'Valores', 'Métricas aditivas sem nulos')

quartis_nan = {c: int(fato[c].isna().sum()) for c in METRICAS
               if ('quartil' in c) and fato[c].isna().any()}
if quartis_nan:
    total = sum(quartis_nan.values())
    registrar('⚠️', 'Valores', 'Quartis suprimidos pela RFB (NaN)',
              f'{total} células em {len(quartis_nan)} coluna(s). Esperado pela LGPD; não imputar.')
else:
    registrar('✅', 'Valores', 'Quartis sem supressão')

relatorio = pd.DataFrame(resultados)
erros = (relatorio['nivel'] == '❌').sum()
avisos = (relatorio['nivel'] == '⚠️').sum()
print(f"Resumo: {len(relatorio)} checagens | ❌ {erros} | ⚠️ {avisos} | ✅ {len(relatorio) - erros - avisos}")
relatorio

## 3 — Reconciliação com a silver

Compara, mês a mês, a soma de `vmle` da gold com a soma direta da silver. Qualquer diferença indica perda ou duplicação na montagem da fato.

In [ ]:
gold_mes = fato.groupby('ano_mes', as_index=False)['vmle_soma_dolar_truncado'].sum()
gold_mes = gold_mes.rename(columns={'vmle_soma_dolar_truncado': 'vmle_gold'})
silver_mes = silver_vmle_por_mes()

rec = gold_mes.merge(silver_mes, on='ano_mes', how='outer')
rec['diferenca'] = (rec['vmle_gold'] - rec['vmle_silver']).round(2)
rec['ok'] = rec['diferenca'].abs() < 0.01

if rec['ok'].all():
    print(f"✅ Reconciliação OK em {len(rec)} meses.")
else:
    print(f"❌ {(~rec['ok']).sum()} mês(es) com diferença:")
rec[~rec['ok']]

## 4 — Exploração inicial

### 4.1 — Série mensal do valor importado

Soma de `vmle` por mês, em bilhões de US$. Lembre-se: a série cobre apenas a parcela publicada (combinações com ≥ 4 importadores), não o total nacional. Oscilações podem refletir também mudanças na publicação, não só no comércio.

In [ ]:
serie = fato.groupby('ano_mes')['vmle_soma_dolar_truncado'].sum().sort_index() / 1e9
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(serie.index, serie.values, marker='o', markersize=3, linewidth=1.5, color='steelblue')
ax.set_title('Valor importado por mês (VMLE, bilhões US$) — parcela publicada pela RFB', loc='left')
ax.set_ylabel('US$ bilhões')
ax.xaxis.set_major_locator(mticker.MaxNLocator(12))
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 4.2 — Principais países de origem (valor acumulado no período)

In [ ]:
por_pais = (fato.groupby('pais_origem')['vmle_soma_dolar_truncado'].sum()
              .sort_values(ascending=False).head(15) / 1e9)
nomes = dim_pais.set_index('pais_origem_rfb')['pais_nome']
rotulos = [nomes.get(p, p) for p in por_pais.index][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
barras = ax.barh(rotulos, por_pais.values[::-1], color='steelblue', edgecolor='white')
ax.bar_label(barras, fmt='%.1f', padding=3)
ax.set_title('Top 15 países de origem — VMLE acumulado (US$ bilhões)', loc='left')
ax.set_xlabel('US$ bilhões')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 4.3 — Principais capítulos NCM (valor acumulado)

O capítulo vem da versão do NCM vigente em cada mês, via `dim_ncm`.

In [ ]:
cap = dim_ncm.drop_duplicates('ncm_versao_id')[['ncm_versao_id', 'capitulo', 'capitulo_descricao']]
fato_cap = fato.merge(cap, on='ncm_versao_id', how='left')
por_cap = (fato_cap.groupby(['capitulo', 'capitulo_descricao'])['vmle_soma_dolar_truncado']
                   .sum().sort_values(ascending=False).head(15) / 1e9).reset_index()
por_cap['rotulo'] = por_cap['capitulo'] + ' · ' + por_cap['capitulo_descricao'].str.slice(0, 45)

fig, ax = plt.subplots(figsize=(11, 6))
barras = ax.barh(por_cap['rotulo'][::-1], por_cap['vmle_soma_dolar_truncado'][::-1],
                 color='darkslateblue', edgecolor='white')
ax.bar_label(barras, fmt='%.1f', padding=3)
ax.set_title('Top 15 capítulos NCM — VMLE acumulado (US$ bilhões)', loc='left')
ax.set_xlabel('US$ bilhões')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 4.4 — Preço médio ponderado por capítulo

Preço médio **recalculado** como `SUM(vmle) / SUM(peso_liq)`. Nunca é média das médias de FOB/kg. Mostra os capítulos com maior valor por quilo, entre os de peso relevante.

In [ ]:
agg = fato_cap.groupby(['capitulo', 'capitulo_descricao'])[['vmle_soma_dolar_truncado', 'peso_liq_soma_truncado']].sum().reset_index()
agg = agg[agg['peso_liq_soma_truncado'] > 0]
agg['usd_por_kg'] = agg['vmle_soma_dolar_truncado'] / agg['peso_liq_soma_truncado']
peso_min = agg['peso_liq_soma_truncado'].quantile(0.5)
top = agg[agg['peso_liq_soma_truncado'] >= peso_min].nlargest(15, 'usd_por_kg').copy()
top['rotulo'] = top['capitulo'] + ' · ' + top['capitulo_descricao'].str.slice(0, 45)

fig, ax = plt.subplots(figsize=(11, 6))
barras = ax.barh(top['rotulo'][::-1], top['usd_por_kg'][::-1], color='seagreen', edgecolor='white')
ax.bar_label(barras, fmt='%.1f', padding=3)
ax.set_title('Preço médio ponderado por capítulo (US$/kg) — top 15, metade mais pesada', loc='left')
ax.set_xlabel('US$ por kg')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 4.5 — Dispersão do FOB/kg por continente

Distribuição da **mediana** de FOB/kg de cada linha (uma linha = NCM × NVE × país × mês), em escala log. Mostra a heterogeneidade de preço dentro de cada continente, sem agregar medianas.

In [ ]:
cont = dim_pais.set_index('pais_origem_rfb')['continente']
d = fato[['pais_origem', 'fob_kg_mediana_dolar_truncado']].dropna()
d = d[d['fob_kg_mediana_dolar_truncado'] > 0].copy()
d['continente'] = d['pais_origem'].map(cont)
ordem = d.groupby('continente')['fob_kg_mediana_dolar_truncado'].median().sort_values().index.tolist()
dados = [d.loc[d['continente'] == c, 'fob_kg_mediana_dolar_truncado'].values for c in ordem]

fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot(dados, tick_labels=ordem, showfliers=False)
ax.set_yscale('log')
ax.set_ylabel('FOB/kg mediana (US$, escala log)')
ax.set_title('Dispersão do preço FOB/kg por continente de origem (sem outliers)', loc='left')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 4.6 — Valor de versões de NCM encerradas × vigentes, por ano

Mostra quanto do valor importado vem de versões de NCM que já foram substituídas (`vigencia_fim` anterior ao sentinela 9999-12-31). Útil para avaliar o impacto das mudanças de classificação na série.

In [ ]:
encerrada = dim_ncm.set_index('ncm_versao_id')['vigencia_fim'] != '9999-12-31'
fato_ano = fato.assign(ano=fato['ano_mes'].str.slice(0, 4),
                       situacao=np.where(fato['ncm_versao_id'].map(encerrada), 'Versão encerrada', 'Versão vigente'))
tab = (fato_ano.groupby(['ano', 'situacao'])['vmle_soma_dolar_truncado'].sum()
               .unstack(fill_value=0) / 1e9)

fig, ax = plt.subplots(figsize=(10, 5))
tab.plot(kind='bar', stacked=True, ax=ax, color=['#bbbbbb', 'steelblue'], edgecolor='white', width=0.7)
ax.set_title('VMLE por ano, segundo a situação da versão do NCM (US$ bilhões)', loc='left')
ax.set_ylabel('US$ bilhões')
ax.set_xlabel('')
ax.legend(title='', frameon=False)
ax.tick_params(axis='x', rotation=0)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5 — Próximos passos

Com as checagens em ✅ e a reconciliação OK, a gold está pronta para os notebooks de insights (`01_padroes_importacao`, `02_analise_temporal`, `03_impacto_regulatorio`). Qualquer ❌ deve ser resolvido na camada anterior, não aqui.